## Sequential

In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
import torchvision.transforms as transforms
from torchvision.datasets import MNIST

In [2]:
# Готовим данные
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])


# Подготовка данных из прошлого урока
dataset = MNIST(root='./data', train=True, download=True, transform=transform)
train_size = int(0.8 * len(dataset))
val_size   = len(dataset) - train_size

train_data, val_data = random_split(dataset, [train_size, val_size])
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=64, shuffle=False)
images, labels = next(iter(train_loader))


# Определяем модель через nn.Sequential

model_seq = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28*28, 256),
    nn.ReLU(),
    nn.Linear(256, 256),
    nn.ReLU(),
    nn.Linear(256, 10)
)


# Прогон и обучение одного батча
out_seq = model_seq(images)
print("Output shape:", out_seq.shape)   # ожидаем [64, 10]


# Оптимизатор и функция потерь
optimizer = torch.optim.AdamW(model_seq.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()


# Прямой проход
loss = criterion(out_seq, labels)


optimizer.zero_grad()
loss.backward()
optimizer.step()

print("Loss после одного шага:", loss.item())

Output shape: torch.Size([64, 10])
Loss после одного шага: 2.295320510864258


---
## Advanced MNIST MLP

In [4]:
import torch
import torch.nn as nn
import torch.nn.init as init

In [6]:
class AdvancedMNISTMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # Определяем слои:
        self.linear1 = nn.Linear(784, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        self.linear2 = nn.Linear(256, 128)
        self.linear3 = nn.Linear(128, 10)
        
        # Инициализация весов слоёв Linear
        init.xavier_uniform_(self.linear1.weight)
        init.xavier_uniform_(self.linear2.weight)
        init.xavier_uniform_(self.linear3.weight)
        # Смещения (bias) по умолчанию инициализируются нулями
        
    def forward(self, x):
        # x ожидаем (batch, 1, 28, 28) или (batch, 784)
        x = x.view(x.size(0), -1)         # разворачиваем изображение
        x = self.linear1(x)
        x = self.bn1(x)                     # нормируем
        x = self.relu(x)
        x = self.dropout(x)                 # регуляризация
        x = self.linear2(x)
        x = self.relu(x)
        x = self.linear3(x)
        return x
    

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
train_data = MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)


# Один батч и прямой проход
model = AdvancedMNISTMLP()


images, labels = next(iter(train_loader))


outputs = model(images)

# Вывод результатов
print("Output shape:", outputs.shape)      # ожидаем [64, 10]
print("Batch labels shape:", labels.shape) # [64]

Output shape: torch.Size([64, 10])
Batch labels shape: torch.Size([64])


---
## Advanced MNIST MLP with Sequential

In [9]:
class AdvancedMNISTMLP(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

        for layer in self.net:
            if isinstance(layer, nn.Linear):
                init.xavier_uniform_(layer.weight)
                layer.bias.data.zero_() 

        
    def forward(self, x):
        self.net(x)
        return x
    

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
train_data = MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)


# Один батч и прямой проход
model = AdvancedMNISTMLP()


images, labels = next(iter(train_loader))


outputs = model(images)

# Вывод результатов
print("Output shape:", outputs.shape)      # ожидаем [64, 10]
print("Batch labels shape:", labels.shape) # [64]

Output shape: torch.Size([64, 1, 28, 28])
Batch labels shape: torch.Size([64])


---
## Advanced MNIST MLP with Module List

In [10]:
class AdvancedMNISTMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.ModuleList([
            nn.Linear(28*28, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        ])

        for layer in self.layers:
            if isinstance(layer, nn.Linear):
                init.xavier_uniform_(layer.weight)
                layer.bias.data.zero_() 

        
    def forward(self, x):
        x = self.flatten(x)
        for layer in self.layers:
            x = layer(x)
        return x
    

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
train_data = MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)


# Один батч и прямой проход
model = AdvancedMNISTMLP()


images, labels = next(iter(train_loader))


outputs = model(images)

# Вывод результатов
print("Output shape:", outputs.shape)      # ожидаем [64, 10]
print("Batch labels shape:", labels.shape) # [64]

Output shape: torch.Size([64, 10])
Batch labels shape: torch.Size([64])
